# Xarray-Spatial KDE: Point density, kernel shapes, and line density

Turning scattered point data into a continuous surface is one of the most common spatial analysis tasks. It comes up whenever you have event locations (earthquakes, crimes, disease cases) and want a smooth density map instead of a dot plot. xrspatial's KDE tools produce density rasters from raw coordinates, with configurable kernels and bandwidth.

### What you'll build

1. Generate synthetic earthquake cluster data
2. Compute a Gaussian density surface with `kde()`
3. Compare three kernel shapes: Gaussian, Epanechnikov, and quartic
4. See how bandwidth controls smoothness
5. Use per-point weights for magnitude-weighted density
6. Map road density with `line_density()`

![KDE preview](images/kde_preview.png)

[Gaussian KDE](#Gaussian-KDE) · [Kernel comparison](#Kernel-comparison) · [Bandwidth effects](#Bandwidth-effects) · [Weighted KDE](#Weighted-KDE) · [Line density](#Line-density)

Standard imports plus the two KDE functions.

In [ ]:
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from xrspatial.kde import kde, line_density

## Synthetic earthquake data

Three clusters of varying tightness and size, plus some background scatter. Each point gets a random magnitude between 2 and 6 that we'll use as weights later.

In [ ]:
rng = np.random.default_rng(42)

# Three clusters with different spreads
c1_x, c1_y = rng.normal(2, 0.8, 120), rng.normal(3, 0.6, 120)
c2_x, c2_y = rng.normal(7, 1.2, 80), rng.normal(7, 1.0, 80)
c3_x, c3_y = rng.normal(5, 0.5, 50), rng.normal(1, 0.5, 50)

# Background scatter
bg_x, bg_y = rng.uniform(0, 10, 30), rng.uniform(0, 10, 30)

x = np.concatenate([c1_x, c2_x, c3_x, bg_x])
y = np.concatenate([c1_y, c2_y, c3_y, bg_y])

# Random magnitudes for weighted KDE later
magnitudes = rng.uniform(2, 6, len(x))

print(f"{len(x)} earthquake epicenters generated")

Raw scatter plot. The three clusters are visible but the exact density pattern is hard to read from points alone.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7.5))
ax.plot(x, y, 'o', color='steelblue', markersize=3, alpha=0.5)
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')
plt.show()

## Gaussian KDE

[Kernel density estimation](https://en.wikipedia.org/wiki/Kernel_density_estimation) places a smooth kernel function at each data point and sums the contributions at every output pixel. The Gaussian kernel is the most common choice because it produces infinitely smooth surfaces with no hard edges.

The plot below shows the density surface with the original points overlaid. Brighter areas have more points per unit area.

In [ ]:
density = kde(x, y, bandwidth=0.6, kernel='gaussian',
              x_range=(0, 10), y_range=(0, 10), width=400, height=400)

fig, ax = plt.subplots(figsize=(10, 7.5))
density.plot.imshow(ax=ax, cmap='inferno', add_colorbar=False)
ax.plot(x, y, 'o', color='white', markersize=1.5, alpha=0.3)
ax.set_title('')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(handles=[Patch(facecolor='yellow', alpha=0.78, label='High density'),
                   Patch(facecolor='#1a0a2e', alpha=0.78, label='Low density')],
          loc='lower right', fontsize=11, framealpha=0.9)
plt.show()

The three clusters show up clearly. The tight cluster in the lower middle produces the sharpest peak, while the spread-out upper-right cluster has a broader, lower dome.

## Kernel comparison

Three kernel shapes are available:

- **Gaussian**: bell curve, infinite range, always smooth
- **Epanechnikov**: parabolic, drops to zero at the bandwidth radius, statistically optimal for minimizing mean integrated squared error
- **Quartic** (biweight): similar compact support but smoother falloff than Epanechnikov

The next plot shows all three side by side on the same data and bandwidth.

In [ ]:
kernels = ['gaussian', 'epanechnikov', 'quartic']
results = {}
for k in kernels:
    results[k] = kde(x, y, bandwidth=0.8, kernel=k,
                     x_range=(0, 10), y_range=(0, 10), width=300, height=300)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, k in zip(axes, kernels):
    results[k].plot.imshow(ax=ax, cmap='inferno', add_colorbar=False)
    ax.set_title(k.capitalize(), fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

The Gaussian surface bleeds out to the edges because it has infinite range. Epanechnikov and quartic cut off at exactly the bandwidth radius, leaving the corners at zero. For most practical work the differences are small; pick Gaussian for smoothness, Epanechnikov if you need a hard cutoff boundary.

## Bandwidth effects

Bandwidth controls how far each point's influence reaches. Too narrow and you get spiky noise; too wide and real structure gets washed out. The default `'silverman'` rule works well for unimodal data but can oversmooth multimodal clusters.

Below: four bandwidths from very narrow (0.2) to very wide (2.0).

In [ ]:
bandwidths = [0.2, 0.5, 1.0, 2.0]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
for ax, bw in zip(axes, bandwidths):
    d = kde(x, y, bandwidth=bw, x_range=(0, 10), y_range=(0, 10),
            width=200, height=200)
    d.plot.imshow(ax=ax, cmap='inferno', add_colorbar=False)
    ax.set_title(f'bw = {bw}', fontsize=12)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

At bw=0.2 you can pick out individual points. At bw=2.0 the three clusters have merged into a single blob. Something around 0.5 to 0.8 captures the cluster structure without too much noise.

<div class="alert alert-block alert-warning">
<b>Units matter.</b> Bandwidth is in the same units as your coordinates. If x and y are in meters, a bandwidth of 500 means 500 m. If they're in degrees, 0.01 means roughly 1 km at mid-latitudes. Always think about what physical distance makes sense for your data before picking a number.
</div>

## Weighted KDE

Not all points are equal. An earthquake of magnitude 5 releases 30x more energy than a magnitude 4. Passing per-point weights lets the density surface reflect importance, not just count.

Below: unweighted (left) vs magnitude-weighted (right). The weighted surface shifts toward wherever the larger events happened to fall.

In [ ]:
unweighted = kde(x, y, bandwidth=0.6,
                 x_range=(0, 10), y_range=(0, 10), width=300, height=300)
weighted = kde(x, y, weights=magnitudes, bandwidth=0.6,
               x_range=(0, 10), y_range=(0, 10), width=300, height=300)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, data, title in zip(axes,
                            [unweighted, weighted],
                            ['Unweighted', 'Magnitude-weighted']):
    data.plot.imshow(ax=ax, cmap='inferno', add_colorbar=False)
    ax.set_title(title, fontsize=13)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xlabel('')
    ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Line density

`line_density()` works the same way but for linear features. Each line segment is uniformly sampled and the samples are convolved with the kernel. The result shows where lines are concentrated, useful for things like road network analysis or fault-line mapping.

Input is four arrays: start x, start y, end x, end y for each segment. The plot below shows a synthetic road grid.

In [ ]:
# Synthetic road segments: a loose grid with some randomness
road_x1, road_y1, road_x2, road_y2 = [], [], [], []

# Horizontal roads
for yy in np.linspace(1, 9, 6):
    offset = rng.normal(0, 0.2)
    road_x1.append(0.5)
    road_y1.append(yy + offset)
    road_x2.append(9.5)
    road_y2.append(yy + rng.normal(0, 0.3))

# Vertical roads (fewer in the east = lower density there)
for xx in np.linspace(1, 5, 6):
    road_x1.append(xx + rng.normal(0, 0.1))
    road_y1.append(0.5)
    road_x2.append(xx + rng.normal(0, 0.1))
    road_y2.append(9.5)

for xx in np.linspace(6, 9, 2):
    road_x1.append(xx)
    road_y1.append(0.5)
    road_x2.append(xx)
    road_y2.append(9.5)

road_x1 = np.array(road_x1)
road_y1 = np.array(road_y1)
road_x2 = np.array(road_x2)
road_y2 = np.array(road_y2)

ld = line_density(road_x1, road_y1, road_x2, road_y2,
                  bandwidth=0.8,
                  x_range=(0, 10), y_range=(0, 10),
                  width=300, height=300)

fig, ax = plt.subplots(figsize=(10, 7.5))
ld.plot.imshow(ax=ax, cmap='inferno', add_colorbar=False)
# Draw the actual road segments on top
for i in range(len(road_x1)):
    ax.plot([road_x1[i], road_x2[i]], [road_y1[i], road_y2[i]],
            color='white', alpha=0.4, linewidth=0.8)
ax.set_title('')
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_xticks([])
ax.set_yticks([])
ax.legend(handles=[Patch(facecolor='yellow', alpha=0.78, label='High road density'),
                   Patch(facecolor='white', alpha=0.4, label='Road segments')],
          loc='lower right', fontsize=11, framealpha=0.9)
plt.show()

The denser grid on the west side shows up clearly as a brighter band. Intersections where horizontal and vertical roads cross produce the brightest hotspots.

<div class="alert alert-block alert-info">
<b>Template grids.</b> Both <code>kde()</code> and <code>line_density()</code> accept a <code>template</code> DataArray instead of <code>x_range</code>/<code>y_range</code>/<code>width</code>/<code>height</code>. The output will match the template's shape, extent, and coordinates. If the template is dask-backed, the output is computed lazily in chunks.
</div>

### References

- [Kernel density estimation](https://en.wikipedia.org/wiki/Kernel_density_estimation), Wikipedia
- [Silverman, B.W. (1986). *Density Estimation for Statistics and Data Analysis*.](https://ned.ipac.caltech.edu/level5/March02/Silverman/paper.pdf) Chapman & Hall.
- [Epanechnikov, V.A. (1969). Non-parametric estimation of a multivariate probability density.](https://doi.org/10.1137/1114019) *Theory of Probability & Its Applications*, 14(1), 153-158.